# Notebook 12 — Learning Graph Cross-Walk (Change B: equivalency-graph-v1)

Marimo walkthrough of the cell-level cross-walk from a single NCCE Y8
Python cell to its equivalents in the 7 other British Isles
jurisdictions:

1. **Ireland** (NCCA Leaving Certificate Computer Science)
2. **England** (AQA GCSE Computer Science)
3. **Wales** (WJEC GCSE Computer Science)
4. **Northern Ireland** (CCEA GCSE Computer Science)
5. **Scotland** (SQA National 5 / Higher Computing Science)
6. **Isle of Man** (Meanscoil CS)
7. **Jersey / Guernsey** (Crown Dependencies)

Powers the
[`2026-08-31-learning-graph-equivalency-graph-v1`](../../openspec/changes/2026-08-31-learning-graph-equivalency-graph-v1/proposal.md)
change. The cross-walk uses BAML
[`ExtractCellEquivalencies`](../../baml_extracts/extract_equivalency.baml)
and materialises to Firestore `prerequisiteEdges/{edge_id}` +
FalkorDB `:CellEquivalentEdge` (dev-deploy path).

In [ ]:
# 1. Verify the source NCCE Y8 Python learning graph exists in dev SQLite.
import pathlib, sqlite3

SQLITE = pathlib.Path("data/bi_ep/extracted_syllabi.sqlite")
print(f"DB exists: {SQLITE.exists()}")
if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        try:
            rows = conn.execute(
                "SELECT id, jurisdiction, subject, year_level, cells_json "
                "FROM learning_graphs "
                "WHERE jurisdiction = 'United Kingdom (NCCE)' "
                "  AND subject = 'computer_science' "
                "  AND year_level = 8 "
                "LIMIT 1"
            ).fetchall()
            print(f"NCCE Y8 CS learning graphs: {len(rows)}")
        except sqlite3.OperationalError as exc:
            print(f"learning_graphs table not present: {exc}")
            print("Run `python -m dlt_pipelines.uk_ncce_learning_graphs` and the `orchestration/defs/3_model_lifecycle/uk_ncce_learning_graphs.py` Dagster assets first.")
else:
    print("No SQLite yet. Pipeline must run first.")


In [ ]:
# 2. Pick one cell from the NCCE Y8 graph (e.g. 'cell_variables_lesson_3' —
# the variable-assignment cell in Lesson 3). In production this comes
# from the Firestore `learningGraphs/{id}` document + a cell-id selector
# dropdown; in dev we hard-code a representative cell.
from baml_client.types import (
    LearningGraph, LearningGraphCell, Jurisdiction, CellEquivalent,
)

source_cell = LearningGraphCell(
    id="cell_variables_lesson_3",
    row_id="row_variables",
    column_id="col_lesson_3",
    skill_description="Assign a value to a named variable in Python 3",
    syntax_code="x = 42",
    pedagogy_principle_ids=["live_coding"],
    bloom_level="REMEMBER_DEFINITION",
    strand="PROGRAMMING",
    confidence=1.0,
)
source_jurisdiction = Jurisdiction.UK_NCCE
target_jurisdictions = [
    Jurisdiction.IRELAND,
    Jurisdiction.ENGLAND,
    Jurisdiction.WALES,
    Jurisdiction.NORTHERN_IRELAND,
    Jurisdiction.SCOTLAND,
    Jurisdiction.ISLE_OF_MAN,
    Jurisdiction.JERSEY,
    Jurisdiction.GUERNSEY,
]
print(f"Source cell: {source_cell.id} — {source_cell.skill_description}")
print(f"Walking to {len(target_jurisdictions)} BI jurisdictions…")


In [ ]:
# 3. Call the BAML `ExtractCellEquivalencies` function for one source cell.
import asyncio
from baml_client import b

async def _walk():
    return await b.ExtractCellEquivalencies(
        source_cell=source_cell,
        source_jurisdiction=source_jurisdiction,
        target_jurisdictions=target_jurisdictions,
    )

try:
    equivalents_map = asyncio.run(_walk())
    n_found = sum(
        1
        for ce in equivalents_map.values()
        if getattr(ce, "cell_id", "")
    )
    print(f"BAML returned {len(equivalents_map)} keys, {n_found} non-empty.")
except Exception as exc:
    print(f"BAML call failed: {exc}")
    equivalents_map = {}
    n_found = 0
    print("(The dev stub returns empty equivalents — set MINIMAX_BASE_URL + MINIMAX_API_KEY for live results.)")


In [ ]:
# 4. Display the cross-walk as a table (jurisdiction -> target cell).
if equivalents_map:
    print(f"{'Jurisdiction':<22}{'Target cell':<38}{'Confidence':<10}{'Notes'}")
    print("-" * 100)
    for j in target_jurisdictions:
        ce = equivalents_map.get(j)
        if ce is None:
            print(f"{str(j):<22}<no result>")
            continue
        cell_id = getattr(ce, "cell_id", "") or "—"
        conf = getattr(ce, "confidence", 0.0)
        notes = getattr(ce, "notes", "") or ""
        marker = ""
        if conf < 0.50:
            marker = "  (low confidence — UI greys this out)"
        print(f"{str(j):<22}{cell_id:<38}{conf:<10.2f}{notes}{marker}")


In [ ]:
# 5. (Optional) Materialise the cross-walk to Firestore + dev SQLite.
# In production this runs as the Dagster asset group:
#   orchestration/defs/3_model_lifecycle/uk_ncce_learning_graph_equivalencies.py
# for each of the 42 (jurisdiction x subject) pairs.
import json
import time
import uuid

edge_id = f"{source_cell.id}__{source_jurisdiction.name}__" + uuid.uuid5(
    uuid.NAMESPACE_DNS, source_cell.id.encode("utf-8")
).hex[:12]
overall_confidence = sum(
    getattr(ce, "confidence", 0.0)
    for ce in equivalents_map.values()
    if getattr(ce, "cell_id", "")
) / max(n_found, 1)
cross_ref = {
    "id": edge_id,
    "source_graph_id": "uk_ncce_y8_intro_to_python",
    "jurisdiction_pair": {"source": str(source_jurisdiction), "target": "MULTI"},
    "cell_edges": [
        {
            "cell_id": getattr(ce, "cell_id", ""),
            "jurisdiction": str(j),
            "subject": "computer_science",
            "year_level": getattr(ce, "year_level", 0),
            "confidence": getattr(ce, "confidence", 0.0),
            "notes": getattr(ce, "notes", ""),
        }
        for j, ce in equivalents_map.items()
    ],
    "overall_confidence": overall_confidence,
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
print(json.dumps(cross_ref, indent=2)[:500], "..." if len(json.dumps(cross_ref)) > 500 else "")


## Summary

- The cell-level cross-walk uses BAML `ExtractCellEquivalencies` to link
  every NCCE cell to its 7 equivalents in the other British Isles
  jurisdictions.
- The cross-walk materialises to Firestore `prerequisiteEdges/{edge_id}`
  and (when FalkorDB is available) to the `:CellEquivalentEdge` graph.
- The Gradio Equivalencies tab visualises the cross-walk as a Sankey
  diagram (Plotly) with confidence scores as link widths.

See [`proposal.md`](../../openspec/changes/2026-08-31-learning-graph-equivalency-graph-v1/proposal.md)
for the full Phase 1-4 plan.